# Attack Zoo, Colab runner

La o sesiune noua se refac pasii 1 si 2.



## 1. Setup


In [ ]:
!git clone https://github.com/Maxxtra/mlsp-attack-zoo.git 2>/dev/null || git -C /content/mlsp-attack-zoo pull
%cd /content/mlsp-attack-zoo
!pip install -q -r requirements.txt
!python src/data.py --download imagenette


## 2. Google Drive

Copiez in Drive rezultatele dupa fiecare model.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Verificarea atacurilor scrise de mana

Comparam FGSM si PGD din `src/my_attacks.py` fata de torchattacks la 1/255, 2/255 si 4/255.
Rulam pe CPU: pe GPU aceleasi convolutii dau rezultate usor diferite intre rulari,
iar acolo unde gradientul e aproape zero semnul se inverseaza.
Trebuie sa iasa `all match`.


In [ ]:
%cd /content/mlsp-attack-zoo
!python src/verify_my_attacks.py --model resnet50 --n 4


## 4. Grila

Per model: fgsm, bim si pgd la 6 bugete, plus deepfool o singura data.
DeepFool cauta perturbatia minima care schimba clasa, deci ignora bugetul.

Un model dureaza intre 10 si 50 de minute, vit fiind cel mai lent. Daca sesiunea
cade, rulam din nou celulele 1 si 2, aducem inapoi rezultatele cu celula 5 si reluam
aici doar modelele care lipsesc.


In [ ]:
%cd /content/mlsp-attack-zoo
for model in ['resnet50', 'efficientnet', 'convnext', 'vit']:
    print(f"\n===== {model} =====")
    !bash scripts/grid.sh {model}
    !cp results/results.csv /content/drive/MyDrive/results.csv


## 5. Ce e gata si ce lipseste

Numara cate bugete are fiecare pereche model-atac: 6 pentru fgsm, bim si pgd, 1 pentru deepfool.
A doua linie aduce inapoi din Drive rezultatele, dupa o sesiune pierduta.


In [ ]:
%cd /content/mlsp-attack-zoo
!cut -d, -f2,3 results/results.csv | sort | uniq -c
# !cp /content/drive/MyDrive/results.csv results/results.csv


## 6. Figura

Se genereaza din `results/results.csv`.


In [ ]:
%cd /content/mlsp-attack-zoo
!python src/make_figures.py
from IPython.display import Image, display
display(Image('figures/robust_acc_imagenette_linf.png'))


## 7. Descarcare


In [ ]:
from google.colab import files
files.download('results/results.csv')
files.download('figures/robust_acc_imagenette_linf.png')
